In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

### Download dos dados

In [ ]:
#Fazer download do dataset e criar o dataloader
train_data = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
train_loader = DataLoader(train_data, batch_size=8, shuffle=True, drop_last=True)

test_data = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())
test_loader = DataLoader(test_data, batch_size=8, shuffle=False, drop_last=False)

Explique o que fazem root, train, download e transform.

O que a função acima (train_data) retornaria sem o transform=transforms.ToTensor?

Isso muda o que foi baixado ou isso muda só o que vai acontece nessa vez? Ou seja, se tirarmos o transform=transforms.ToTensor(), teremos que baixar os dados de novo?

---------------------

O que são o batch_size e o shuffle? Por que você poderia querer o shuffle?

O que faz o drop_last=True? Por que você poderia querer isso?

O que faz o data_loader?

In [ ]:
### ZONA DE TESTES ###

In [ ]:
# 1. Crie o iterador
dataiter = iter(train_loader)

# 2. Use a função next() para pegar o primeiro batch
images, labels = next(dataiter)

fig, axes = plt.subplots(1, 6, figsize=(12, 2))

for i in range(6):
    # .permute(1, 2, 0) é útil caso a imagem seja colorida (RGB)
    # .squeeze() remove dimensões unitárias (ex: escala de cinza)

    img = images[i].squeeze()
    print(f'Original shape: {images[i].shape}, After squeeze: {img.shape}')
        
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f'Label: {labels[i].item()}')
    axes[i].axis('off')

plt.show()

O que está acontecendo acima?


- Descubra o que next(iter(train_loader)) retorna. 

- Verifique o fromato de next(iter(train_loader))[i], para i $\in$ {0,1}.
O que são essas coisas?

- Por que fizemos a transformação .sequeeze()? O que ela fez? Poderíamos não ter feito ela? (tire ela e teste, explique o que aconteceu)

In [ ]:
### ZONA DE TESTES ###

Argumente aqui que fazer rotações (grandes) nesse dataset parece ser uma ideia ruim. Em particular, argumente que rotações de -90 ou 90 graus já traríam problemas intratáveis.

---------------------

### Alguns testes:

- Faça uma análise sobre os tamanhos/dimensôes das imagens. O que podemos concluir? Isso é bom para nós de alguma forma?

- Faça um histograma das classes (isto é, queremos ver o quão balanceado é o dataset (de treino)). Conclua algo.

In [ ]:
### ZONA DE TESTES ###

### Criando um loop:

(tente endender bem os elementos do loop a seguir, ele irá aparecer depois)

In [ ]:
for epoch in range(2):
    for i, data in enumerate(train_loader, 0): #o que aconteceria sem o enumerate? E o 0?
        inputs, labels = data
        
        # aqui você faria o forward pass, backward pass e otimização

        if i == 0:  # Mostrando apenas o primeiro batch de cada época
            print(f'Epoch {epoch+1}:')
            print(f'Inputs shape: {inputs.shape}')
            print(f'Labels shape: {labels.shape}')
            print('\n')
        


- Por que o 8 aparece no size? Onde isso foi definido?

### Modelo

Faremos um modelo muito simples. Faremos um MLP em vez de usar uma CNN, para não entrar no território do trabalho 1.

In [ ]:
# define a MLP
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28*28, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.relu = nn.ReLU()
        
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.bn1(x)
        x = self.fc2(x)
        return x

### Perdas

Pesquise sobre essas perdas:

- nn.MSELoss
- nn.CrossEntropyLoss
- nn.BCELoss
- nn.L1Loss

Dê um exemplo de onde cada poderia ser usada e escolha uma para usar no nosso treinamento. (a seguir)


### Treinando
#### (juntando o modelo e o loop)

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

model = MLP().to(device)
criterion = nn.CrossEntropyLoss()  # Definindo a função de perda para classificação
if criterion is None:
    raise ValueError("O critério de perda (loss function) não foi definido. Por favor, defina uma função de perda adequada para o problema de classificação.")

optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [ ]:
#Mudei o batchsize para 64 para acelerar o treinamento, mas isso só deu certo 
train_data = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, drop_last=True)

test_data = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())
test_loader = DataLoader(test_data, batch_size=64, shuffle=False, drop_last=False)

In [ ]:
train_losses = []
test_losses = []

for epoch in range(10):
    for i, data in enumerate(train_loader, 0):

        print(f'Epoch {epoch+1}, Batch {i+1}', end='\r')  # Printando o progresso do treinamento
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)
        # forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

        # test loss
        model.eval()
        with torch.no_grad():
            test_loss = 0
            for data in test_loader:
                inputs, labels = data
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss += loss.item()
            test_losses.append(test_loss / len(test_loader))
        model.train()

    # save the model
    if (epoch + 1) % 10 == 0:  # Salvando o modelo a cada 10 épocas
        torch.save(model.state_dict(), f'mnist_mlp_{epoch}.pth')

- O que fez o model.eval() ? E o model.train()?
- O que fizeram as partes .to(device)?

In [ ]:
# plotting learning curves
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(test_losses, label='Test Loss')
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.legend()

## Resultados:

O modelos claramente não está muito bom... Ele talvez seja muito pequeno para capturar os dados, ele também quebra a geometria dos dados quando os planifica (uma CNN não faria isso!)
Poderíamos trainar por mais épocas, o que ajudaria nos resultados, mas, nesse caso, não muito significativamente.

Caso você já esteja confortável com o que fizemos nesse notebook, tente fazer o trablho 1! Nele teremos que fazer uma CNN para classificar alguns animais quanto a sua espécie e raça.

Caso ainda esteja com dúvidas, tente aumentar o nosso modelo para que ele comece a classificar melhor os números! ;)